# Using PySpark

In [77]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet("../data/cleaned/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

# df = df.filter(F.col("Geopolitical entity (reporting)") == "Austria")
df = df.filter(F.col("Motor energy") == "Petrol (excluding hybrids)")

# df.show(5, truncate=False)

Get the number of unique combinations of <'commercial_name', 'engine_capacity (cm3)', and 'engine_power (KW)'> for the consumer for each <'TIME_PERIOD', 'Motor energy'>.

In [78]:
unique_consumer_choices_counts = (
    df
    .groupBy(
        "TIME_PERIOD",
        "Motor energy"
    )
    .agg(
        F.countDistinct(
            F.struct(
                "commercial_name",
                "engine_capacity (cm3)",
                "engine_power (KW)"
            )
        ).alias("unique_choices")
    )
    .orderBy(
        "TIME_PERIOD",
        "Motor energy"
    )
)

unique_consumer_choices_counts.show(20, truncate=False)

+-----------+--------------------------+--------------+
|TIME_PERIOD|Motor energy              |unique_choices|
+-----------+--------------------------+--------------+
|2014       |Petrol (excluding hybrids)|8467          |
|2015       |Petrol (excluding hybrids)|8207          |
|2016       |Petrol (excluding hybrids)|7595          |
|2017       |Petrol (excluding hybrids)|8527          |
|2018       |Petrol (excluding hybrids)|7626          |
|2019       |Petrol (excluding hybrids)|5782          |
|2020       |Petrol (excluding hybrids)|5215          |
|2021       |Petrol (excluding hybrids)|5629          |
|2022       |Petrol (excluding hybrids)|4852          |
|2023       |Petrol (excluding hybrids)|9981          |
+-----------+--------------------------+--------------+



Compute baseline metric

In [79]:
import pyspark.sql.functions as F

by_motor = (
    df
    .groupBy("TIME_PERIOD", "Motor energy")
    .agg(
        F.countDistinct(
            F.struct(
                "commercial_name",
                "engine_capacity (cm3)",
                "engine_power (KW)"
            )
        ).alias("unique_choices"),
        F.sum("registrations").alias("registrations_count")
    )
)

# all_motor = (
#     by_motor
#     .groupBy("TIME_PERIOD")
#     .agg(
#         F.sum("unique_choices").alias("unique_choices"),
#         F.sum("registrations_count").alias("registrations_count")
#     )
#     .withColumn("Motor energy", F.lit("All"))
# )

unique_consumer_choices_counts = (
    by_motor
    # .unionByName(all_motor)
    .withColumn(
        "baseline_normalized_registrations", 
        F.col("registrations_count") / F.col("unique_choices")
    )
    .orderBy("TIME_PERIOD", "Motor energy")
)

unique_consumer_choices_counts.show(20, truncate=False)

+-----------+--------------------------+--------------+-------------------+---------------------------------+
|TIME_PERIOD|Motor energy              |unique_choices|registrations_count|baseline_normalized_registrations|
+-----------+--------------------------+--------------+-------------------+---------------------------------+
|2014       |Petrol (excluding hybrids)|8467          |4314669            |509.58651234203376               |
|2015       |Petrol (excluding hybrids)|8207          |4904806            |597.6368953332521                |
|2016       |Petrol (excluding hybrids)|7595          |5538117            |729.1793285055958                |
|2017       |Petrol (excluding hybrids)|8527          |6491625            |761.30233376334                  |
|2018       |Petrol (excluding hybrids)|7626          |7496533            |983.0229478101232                |
|2019       |Petrol (excluding hybrids)|5782          |8021718            |1387.3604289173297               |
|2020     

# Without PySpark, you can achieve this using pandas.

In [80]:
import pandas as pd

df = pd.read_parquet("../data/cleaned/4c_eea_co2_emissions_from_passenger_cars-001.parquet", engine="pyarrow")

df = df[df["Geopolitical entity (reporting)"] == "Austria"]
df = df[df["Motor energy"] == "Petrol (excluding hybrids)"]

# display(df.head(5))

Get the number of unique combinations of <'commercial_name', 'engine_capacity (cm3)', and 'engine_power (KW)'> for the consumer for each <'TIME_PERIOD', 'Motor energy'>.

In [81]:
unique_consumer_choices_counts = (
    df.drop_duplicates(subset=[
        "TIME_PERIOD", 
        "Motor energy", 
        "commercial_name", 
        "engine_capacity (cm3)", 
        "engine_power (KW)"
    ])
    .groupby(["TIME_PERIOD", "Motor energy"])
    .size()
    .reset_index(name="unique_choices")
)

display(unique_consumer_choices_counts)

,TIME_PERIOD,Motor energy,unique_choices
0,2014,Petrol (excluding hybrids),1419
1,2015,Petrol (excluding hybrids),1526
2,2016,Petrol (excluding hybrids),1463
3,2017,Petrol (excluding hybrids),1528
4,2018,Petrol (excluding hybrids),1785
5,2019,Petrol (excluding hybrids),1508
6,2020,Petrol (excluding hybrids),1532
7,2021,Petrol (excluding hybrids),1568
8,2022,Petrol (excluding hybrids),1149
9,2023,Petrol (excluding hybrids),1043


Compute baseline metric

In [82]:
registrations_counts = (
    df.groupby(["TIME_PERIOD", "Motor energy"])["registrations"]
    .sum()
    .reset_index(name="registrations_count")
)

unique_consumer_choices_counts = unique_consumer_choices_counts.merge(
    registrations_counts, 
    on=["TIME_PERIOD", "Motor energy"]
)

unique_consumer_choices_counts["baseline_normalized_registrations"] = (
    unique_consumer_choices_counts["registrations_count"] / unique_consumer_choices_counts["unique_choices"]
)

display(unique_consumer_choices_counts)

,TIME_PERIOD,Motor energy,unique_choices,registrations_count,baseline_normalized_registrations
0,2014,Petrol (excluding hybrids),1419,128250,90.380550
1,2015,Petrol (excluding hybrids),1526,124718,81.728702
2,2016,Petrol (excluding hybrids),1463,135040,92.303486
3,2017,Petrol (excluding hybrids),1528,170212,111.395288
4,2018,Petrol (excluding hybrids),1785,190225,106.568627
5,2019,Petrol (excluding hybrids),1508,186673,123.788462
6,2020,Petrol (excluding hybrids),1532,124749,81.428851
7,2021,Petrol (excluding hybrids),1568,120639,76.938138
8,2022,Petrol (excluding hybrids),1149,106603,92.778938
9,2023,Petrol (excluding hybrids),1043,113868,109.173538
